# Convergence of average error for the number of detected sources
First we find the minimal separation distance for LASSO, Sparse Prior and the pseudoinverse solution $X_0 = A^{\dagger}Y$, then we create as many sources as can fit within a circle of that spacing and finally we run enough simulations that the mean errors converge.

In [ ]:
from compare_sparsity_LASSO import *
from cs_priors.plotting.plotting import (
    plot_equation,
    plot_two_line_equation,
    plot_overview,
    wrapper_plot_geometry,
)

Find the minimum distance between the elements

In [ ]:
sim = run_simulation(num_mics=8, num_sources=2, s_sparse=2, angle_step=1e-10)
X0 = np.linalg.pinv(sim.C[:,:,1]) @ sim.Y[:,1]
X_TRUE = sim.X[:,1]
X_sp, B = sparse_prior_solution(X0, sim.C[:,:,1])
plot_overview(sim)
plot_equation(X_TRUE, X_sp.reshape(-1,1) - X_TRUE.reshape(-1,1), X0 - X_TRUE, (r"$X_{True}$", r"$X_{True}-X_{sp}$", r"$X_{True}-X_0$"))
# plot_equation(sim.Y[:,1], sim.C[:,:,1], X_TRUE, (r"$Y$", r"$A$", r"$X_{True}$"))

In [ ]:
angle_steps = np.logspace(-9, -10, 30)
num_mics = 8
num_sources = 2

error_lasso = np.zeros(len(angle_steps))
error_sp = np.zeros(len(angle_steps))
error_X0 = np.zeros(len(angle_steps))

num_seeds = 20
for seed in range(num_seeds):
    np.random.seed(seed)
    for i, angle_step in enumerate(angle_steps):
        angle_base = np.random.uniform(0, 360)
        phase_step = np.random.uniform(0, 2 * np.pi)
        Y, A, X0, X_TRUE = just_YAX_from_simulation(num_mics=num_mics, num_sources=num_sources, s_sparse=num_sources, angle_step=angle_step, angle_base=angle_base, phase_step=phase_step)
        X_sp, B = sparse_prior_solution(X0, A)
        X_lasso = complex_lasso(A, Y, alpha=1e-3)

        error_lasso[i] = np.linalg.norm(X_lasso.reshape(-1, 1) - X_TRUE.reshape(-1, 1)) / np.linalg.norm(X_TRUE)
        error_sp[i] = np.linalg.norm(X_sp.reshape(-1, 1) - X_TRUE.reshape(-1, 1)) / np.linalg.norm(X_TRUE)
        error_X0[i] = np.linalg.norm(X0.reshape(-1, 1) - X_TRUE.reshape(-1, 1)) / np.linalg.norm(X_TRUE)
    
    sp_marker_size = 5
    # only set the label once
    if seed == num_seeds - 1:
        plt.plot(angle_steps, error_X0, label="Pseudo-inverse", color="gray", linestyle="--")
        plt.scatter(angle_steps, error_lasso, label="LASSO", color="orange")
        plt.scatter(angle_steps, error_sp, label="Sparse Prior", color="blue", marker="x", s=30)
    else:
        plt.plot(angle_steps, error_X0, color="gray", linestyle="--")
        plt.scatter(angle_steps, error_lasso, color="orange")
        plt.scatter(angle_steps, error_sp, color="blue", marker="o", s=sp_marker_size)

plt.xscale("log")
plt.xlabel("Angle step (degrees)")
plt.ylabel("Relative error")
plt.grid(True, which="both", ls="--")
plt.legend()
plt.title("Reconstruction error vs angle step")
plt.show()

# Minimal angle setup
Sparse prior has worse resolution than the pseudoinverse, but can handle an angle difference of $10^{-9}$ radians.
Rounding $\frac{2 \pi}{10^{-9}}$ we have at least $ 6 \ 283 \ 185 \ 307$ angles, so the limiting factor is computational power, not angles.

We explore 100 angles, i.e. $0.06283$ angle difference over a full circle.

In [ ]:
num_sources = 100

sim = run_simulation(num_mics=8, num_sources=num_sources, s_sparse=2, angle_step=2*np.pi/num_sources)
wrapper_plot_geometry(sim, figsize=(16,16), fontsize=16, skip_labels=True)
Y = sim.Y[:,1]
A = sim.C[:,:,1]
X0 = np.linalg.pinv(A) @ Y
X_TRUE = sim.X[:,1]
X_sp, B = sparse_prior_solution(X0, A)
plot_equation(X_TRUE, X_sp.reshape(-1,1) - X_TRUE.reshape(-1,1), X0 - X_TRUE, (r"$X_{True}$", r"$X_{True}-X_{sp}$", r"$X_{True}-X_0$"))

Is the sparse prior method too slow?

In [ ]:
import time

def covariance_matrices(
    num_sources: int, variance: float = 1.0, spread: float = 0.005
) -> list[np.ndarray]:
    """Returns diagonal elements of each covariance matrix"""
    return [
        np.array([variance if j == i else spread for j in range(num_sources)])
        for i in range(num_sources)
    ]

def precompute_hessian_constant(B_real, D_diags):
    """Precompute B.T @ D_i @ B for all diagonal matrices D_i"""
    return [B_real.T @ np.diag(d) @ B_real for d in D_diags]

def first_derivative_objective(z: np.ndarray) -> np.ndarray:
    # (-1) e^{-(x_0 + Bz)^T D (x_0+Bz)}(B^T(D + D^T) (x_0 + B z))
    # where D_i is symmetric
    x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
    x_flat = x.flatten()
    grad = np.zeros_like(z)
    for d_i in D_diags:
        # Diagonal matrix multiply: D @ x = d * x (element-wise)
        qf = np.dot(x_flat * d_i, x_flat)  # x.T @ D @ x where D is diagonal
        exp_neg_qf = np.exp(-qf)
        # D @ x as element-wise multiply, then B.T @ (D @ x)
        grad += 2 * exp_neg_qf * (B_real.T @ (d_i * x_flat))
    return grad

def second_derivative_objective(z: np.ndarray) -> np.ndarray:
    x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
    x_flat = x.flatten()
    hess = np.zeros((z.shape[0], z.shape[0]))
    for i, d_i in enumerate(D_diags):
        qf = np.dot(x_flat * d_i, x_flat)
        exp_neg_qf = np.exp(-qf)
        # B.T @ (D @ x) where D is diagonal
        BdX = B_real.T @ (d_i * x_flat)
        # Use precomputed B.T @ D @ B
        hess += exp_neg_qf * (2 * np.outer(BdX, BdX) - 2 * BtDB_precomputed[i])
    return -hess

def optimize_objective(X0_real, B_real, D_diags, callback=None, z_start=None):
    def negative_objective(z: np.ndarray) -> float:
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        x_flat = x.flatten()
        return -sum(np.exp(-np.dot(x_flat * d_i, x_flat)) for d_i in D_diags)

    if z_start is None:
        z_start = np.zeros(B_real.shape[1])

    start_time = time.time()
    res = optimize.minimize(
        negative_objective,
        z_start,
        method='l-BFGS-B',
        callback=callback,
        jac=first_derivative_objective,
    )
    end_time = time.time()
    print(f"Optimization time: {end_time - start_time:.4f} seconds")
    
    z_opt = res.x
    x_opt = X0_real.reshape(-1, 1) + (B_real @ z_opt.reshape(-1, 1))
    x_opt_complex = from_real_augmented(x_opt)
    return z_opt, x_opt, x_opt_complex, res

U, S, Vt = np.linalg.svd(A)
rank = np.sum(S > 1e-10)

if rank == A.shape[1]:
    print("No null space detected.")

B = Vt[rank:].T
B_real = np.block([[B.real, -B.imag], [B.imag, B.real]])
X0_real = to_real_augmented(X0)
D_diags = covariance_matrices(num_sources=X0_real.shape[0])

# Precompute constant Hessian terms (only if using Hessian)
BtDB_precomputed = precompute_hessian_constant(B_real, D_diags)

z_opt, x_opt, x_opt_complex, res = optimize_objective(X0_real, B_real, D_diags)